# Version 1 — Cyclone Intensification Prediction Using IBTrACS
Predicts Bay of Bengal cyclone intensification at 48–72 hours and 120–168 hours using historical track features only.

In [ ]:
!pip -q install pandas numpy scikit-learn xgboost matplotlib seaborn

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)
from xgboost import XGBClassifier

print('Imports OK')

In [ ]:
# ── Load IBTrACS ──────────────────────────────────────────────────────────────
IBTRACS_URL = (
    'https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs'
    '/v04r01/access/csv/ibtracs.NI.list.v04r01.csv'
)

print('Loading IBTrACS …')
df_raw = pd.read_csv(IBTRACS_URL, skiprows=[1], low_memory=False)
print(f'Raw shape: {df_raw.shape}')

# Coerce numeric columns
for col in ['LAT', 'LON', 'USA_WIND', 'WMO_WIND', 'USA_PRES']:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

# Parse ISO_TIME as UTC
df_raw['ISO_TIME'] = pd.to_datetime(df_raw['ISO_TIME'], utc=True, errors='coerce')

# Build unified WIND column
if 'USA_WIND' in df_raw.columns and 'WMO_WIND' in df_raw.columns:
    df_raw['WIND'] = df_raw['USA_WIND'].fillna(df_raw['WMO_WIND'])
elif 'USA_WIND' in df_raw.columns:
    df_raw['WIND'] = df_raw['USA_WIND']
elif 'WMO_WIND' in df_raw.columns:
    df_raw['WIND'] = df_raw['WMO_WIND']
else:
    df_raw['WIND'] = np.nan

df_raw['WIND'] = pd.to_numeric(df_raw['WIND'], errors='coerce')

# Drop rows missing essentials
df_raw = df_raw.dropna(subset=['SID', 'ISO_TIME', 'WIND'])
df_raw = df_raw.sort_values(['SID', 'ISO_TIME']).reset_index(drop=True)

# Filter Bay of Bengal: LAT 0–25, LON 80–100
df = df_raw[
    df_raw['LAT'].between(0, 25) & df_raw['LON'].between(80, 100)
].reset_index(drop=True)

print(f'Bay of Bengal rows: {len(df)}, unique storms: {df["SID"].nunique()}')
df[['SID', 'ISO_TIME', 'LAT', 'LON', 'WIND']].head()

In [ ]:
# ── Track Feature Engineering ─────────────────────────────────────────────────
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2)**2
    return R * 2 * math.asin(math.sqrt(max(0.0, min(1.0, a))))


def add_track_features(df):
    df = df.copy().sort_values(['SID', 'ISO_TIME']).reset_index(drop=True)

    df['MONTH'] = df['ISO_TIME'].dt.month.astype(float)
    df['HOUR']  = df['ISO_TIME'].dt.hour.astype(float)

    grp = df.groupby('SID', sort=False)
    df['LAT_CHANGE']  = grp['LAT'].diff()
    df['LON_CHANGE']  = grp['LON'].diff()
    df['WIND_CHANGE'] = grp['WIND'].diff()

    speeds = np.full(len(df), np.nan)
    for sid, group in df.groupby('SID', sort=False):
        idxs  = group.index.tolist()
        lats  = group['LAT'].values
        lons  = group['LON'].values
        times = group['ISO_TIME'].values
        for k in range(1, len(idxs)):
            try:
                dist_km = haversine_km(lats[k-1], lons[k-1], lats[k], lons[k])
                dt_hr   = (times[k] - times[k-1]) / np.timedelta64(1, 'h')
                if dt_hr > 0:
                    speeds[idxs[k]] = dist_km / dt_hr
            except Exception:
                pass
    df['TRACK_SPEED'] = speeds
    return df


df = add_track_features(df)
print('Features added:', ['MONTH','HOUR','LAT_CHANGE','LON_CHANGE','WIND_CHANGE','TRACK_SPEED'])
df[['MONTH','HOUR','LAT_CHANGE','LON_CHANGE','WIND_CHANGE','TRACK_SPEED']].describe()

In [ ]:
# ── Label Creation ────────────────────────────────────────────────────────────
def create_labels(df, start_hours, end_hours, threshold=15):
    """Binary intensification label for a given forecast window."""
    df = df.copy().sort_values(['SID', 'ISO_TIME']).reset_index(drop=True)
    targets = np.full(len(df), np.nan)

    for sid, group in df.groupby('SID', sort=False):
        idxs  = group.index.tolist()
        times = group['ISO_TIME'].values
        winds = group['WIND'].values

        for pos, i in enumerate(idxs):
            t = times[pos]
            w_curr = winds[pos]
            dt_hours = (times - t) / np.timedelta64(1, 'h')
            mask = (dt_hours >= start_hours) & (dt_hours <= end_hours)
            future_winds = winds[mask]

            if len(future_winds) == 0:
                targets[i] = np.nan
            else:
                max_future = float(np.nanmax(future_winds))
                targets[i] = 1.0 if (max_future - w_curr) >= threshold else 0.0

    df['TARGET'] = targets
    return df


# Short horizon: 48–72 h
df_short = create_labels(df, start_hours=48, end_hours=72)
print('\n--- Short horizon (48–72 h) ---')
print(df_short['TARGET'].value_counts(dropna=False))

# Long horizon: 120–168 h
df_long = create_labels(df, start_hours=120, end_hours=168)
print('\n--- Long horizon (120–168 h) ---')
print(df_long['TARGET'].value_counts(dropna=False))

In [ ]:
# ── Training & Evaluation ─────────────────────────────────────────────────────
FEATURES = ['LAT', 'LON', 'WIND', 'MONTH', 'HOUR',
            'LAT_CHANGE', 'LON_CHANGE', 'WIND_CHANGE', 'TRACK_SPEED']


def train_and_evaluate(df_labeled, horizon_label=''):
    """Train RF and XGBoost on labeled data; return fitted XGB pipeline."""
    df_clean = df_labeled.dropna(subset=['TARGET'] + [c for c in FEATURES if c in df_labeled.columns])
    # keep only rows with enough non-null features
    feat_cols = [c for c in FEATURES if c in df_clean.columns]
    df_clean = df_clean.dropna(subset=['TARGET'])

    X = df_clean[feat_cols].values
    y = df_clean['TARGET'].astype(int).values
    groups = df_clean['SID'].values

    unique_storms = np.unique(groups)
    if len(unique_storms) < 2:
        print(f'[{horizon_label}] Not enough storms for split — skipping.')
        return None, None

    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups))

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    print(f'\n{"=" * 60}')
    print(f'  {horizon_label}')
    print(f'  Train: {len(y_train)} rows | Test: {len(y_test)} rows')
    print(f'  Train class dist: {dict(zip(*np.unique(y_train, return_counts=True)))}')
    print(f'  Test  class dist: {dict(zip(*np.unique(y_test,  return_counts=True)))}')

    models = {
        'RandomForest': Pipeline([
            ('imputer',    SimpleImputer(strategy='median')),
            ('classifier', RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')),
        ]),
        'XGBoost': Pipeline([
            ('imputer',    SimpleImputer(strategy='median')),
            ('classifier', XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                         use_label_encoder=False, eval_metric='logloss',
                                         random_state=42, verbosity=0)),
        ]),
    }

    xgb_pipe = None
    for name, pipe in models.items():
        pipe.fit(X_train, y_train)
        y_prob = pipe.predict_proba(X_test)[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)
        print(f'\n  ── {name} ──')
        print(classification_report(y_test, y_pred, zero_division=0))
        print('  Confusion matrix:')
        print(confusion_matrix(y_test, y_pred))
        if len(np.unique(y_test)) >= 2:
            print(f'  ROC-AUC : {roc_auc_score(y_test, y_prob):.4f}')
            print(f'  PR-AUC  : {average_precision_score(y_test, y_prob):.4f}')
        else:
            print('  [WARNING] Only one class in test set — skipping ROC/PR-AUC')
        print(f'  Brier   : {brier_score_loss(y_test, y_prob):.4f}')
        if name == 'XGBoost':
            xgb_pipe = pipe

    return xgb_pipe, feat_cols


xgb_short, feat_short = train_and_evaluate(df_short, 'SHORT HORIZON (48–72 h)')
xgb_long,  feat_long  = train_and_evaluate(df_long,  'LONG HORIZON (120–168 h)')

In [ ]:
# ── Feature Importance ────────────────────────────────────────────────────────
def plot_feature_importance(pipeline, feature_names, title='Feature Importance'):
    if pipeline is None:
        print(f'No model available for: {title}')
        return
    clf = pipeline.named_steps['classifier']
    importances = clf.feature_importances_
    n = len(importances)
    names = list(feature_names)[:n]
    order = np.argsort(importances)
    fig, ax = plt.subplots(figsize=(8, max(4, n * 0.4)))
    ax.barh(np.array(names)[order], importances[order], color='steelblue')
    ax.set_xlabel('Importance')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


plot_feature_importance(xgb_short, feat_short or FEATURES, 'XGBoost Feature Importance — Short Horizon (48–72 h)')
plot_feature_importance(xgb_long,  feat_long  or FEATURES, 'XGBoost Feature Importance — Long Horizon (120–168 h)')

In [ ]:
# ── Example Prediction ────────────────────────────────────────────────────────
# Demonstrate how to run predict_proba on a new observation.

example_obs = {
    'LAT': 15.0,
    'LON': 88.5,
    'WIND': 45.0,       # knots
    'MONTH': 10.0,
    'HOUR': 6.0,
    'LAT_CHANGE': 0.3,
    'LON_CHANGE': -0.2,
    'WIND_CHANGE': 5.0,
    'TRACK_SPEED': 12.0,
}

feat_cols = feat_short or FEATURES
X_new = pd.DataFrame([example_obs])[feat_cols].values

if xgb_short is not None:
    prob_short = xgb_short.predict_proba(X_new)[0, 1]
    print(f'Short-horizon (48–72 h) intensification probability : {prob_short:.3f}')

if xgb_long is not None:
    feat_cols_long = feat_long or FEATURES
    X_new_long = pd.DataFrame([example_obs])[feat_cols_long].values
    prob_long = xgb_long.predict_proba(X_new_long)[0, 1]
    print(f'Long-horizon  (120–168 h) intensification probability: {prob_long:.3f}')